# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202302_Earthquake_Turkiye'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'landsat'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = {
    "driver": "COG",
    "compress": "DEFLATE",
}

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [7]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 91 .tif files in the S3 bucket.


['drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_174034_20230222_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_174035_20230222_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_176034_20230220_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_176035_20230220_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171034_20230116_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171034_20230116_trueColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171035_20230116_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171035_20230116_trueColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_172034_20230208_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiy

# For these we can see three different types of files

We will use the same rename function and place them into the same directory


## Configure bucket and paths (no need to create session manually)

In [8]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [9]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 28
  - Total size: 3.82 GB

📁 Cached files (first 10):
  - drcs_activations/202302_Earthquake_Turkiye/dnb/20230103_dnbrgb.tif (33.0 MB)
  - drcs_activations/202302_Earthquake_Turkiye/dnb/20230208_dnbrgb.tif (33.0 MB)
  - drcs_activations/202302_Earthquake_Turkiye/eos_rs/EOS-RS_20230208_DPM_A2_Türkiye_Syria_Earthquake_v0.5.tif (138.7 MB)
  - drcs_activations/202302_Earthquake_Turkiye/eos_rs/EOS-RS_20230208_DPM_A2_Türkiye_Syria_Earthquake_v0.5_cvd.tif (138.7 MB)
  - drcs_activations/202302_Earthquake_Turkiye/eos_rs/EOS-RS_20230209_DPM_S1_Turkiye_Syria_Earthquake_v0.9.tif (711.1 MB)
  - drcs_activations/202302_Earthquake_Turkiye/eos_rs/EOS-RS_20230209_DPM_S1_Turkiye_Syria_Earthquake_v0.9_cvd.tif (711.1 MB)
  - drcs_activations/202302_Earthquake_Turkiye/eos_rs/EOS-RS_20230210_DPM_S1_Turkiye_Syria_Earthquake_v0.9.tif (446.9 MB)
  - drcs_activations/202302_Earthquake_Turkiye/eos_rs/EOS-RS_20230210_DPM_S1_Turkiye_Syria_Earthquak

(28, 4098579340)

In [10]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [11]:
keys

['drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_174034_20230222_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_174035_20230222_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_176034_20230220_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_176035_20230220_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171034_20230116_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171034_20230116_trueColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171035_20230116_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171035_20230116_trueColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_172034_20230208_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiy

# Landsat 8, trueColor

In [17]:
# Define filename creator functions for different file types

def create_cog_filename(f, EVENT_NAME):
    """Extract date from filename and move to end with formatted date."""
    from pathlib import Path
    import re
    
    filename_stem = Path(f).stem
    
    # Find date pattern (8 digits starting with 20)
    date_match = re.search(r'(20\d{6})', filename_stem)
    
    if date_match:
        date_str = date_match.group(1)
        # Format date as YYYY-MM-DD
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Remove the date from its current position
        filename_parts = filename_stem.replace(date_str + '_', '')
        
        # Create new filename with EVENT_NAME + parts + formatted date + day
        cog_filename = f'{EVENT_NAME}_{filename_parts}_{formatted_date}_day.tif'
    else:
        # No date found, just add event name
        cog_filename = f'{EVENT_NAME}_{filename_stem}_day.tif'
    
    return cog_filename


pattern = re.compile(r'LC08.*trueColor\.tif$')

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if pattern.search(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202302_Earthquake_Turkiye_LC08_L2SP_171034_trueColor_2023-01-16_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_171035_trueColor_2023-01-16_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172034_trueColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172035_trueColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172036_trueColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174033_trueColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174034_trueColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174035_trueColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174036_trueColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_175034_trueColor_2023-02-13_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_175035_trueColor_2023-02-13_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_176034_trueColor_2023-01-19_day.tif


In [18]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename, 
                                target_dir = "Landsat/trueColor", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202302_Earthquake_Turkiye_LC08_L2SP_171034_trueColor_2023-01-16_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_171035_trueColor_2023-01-16_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172034_trueColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172035_trueColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172036_trueColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174033_trueColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174034_trueColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174035_trueColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174036_trueColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_175034_trueColor_2023-02-13_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_175035_trueColor_2023-02-13_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_176034_trueColor_2023-01-19_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/20230

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/202302_Earthquake_Turkiye_LC08_L2SP_171034_trueColor_2023-01-16_day.tif
   [MEMORY] Final: 575.3 MB (Change: +276.5 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC08_L2SP_171034_trueColor_2023-01-16_day.tif

[2/12] Processing: drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171035_20230116_trueColor.tif
   Output filename: 202302_Earthquake_Turkiye_LC08_L2SP_171035_trueColor_2023-01-16_day.tif
   [MEMORY] Initial: 575.3 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJEC

   [BAND 2/3] Processing...


...

   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/202302_Earthquake_Turkiye_LC08_L2SP_175035_trueColor_2023-02-13_day.tif
   [MEMORY] Final: 693.8 MB (Change: -10.7 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC08_L2SP_175035_trueColor_2023-02-13_day.tif

[12/12] Processing: drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_176034_20230119_trueColor.tif
   Output filename: 202302_Earthquake_Turkiye_LC08_L2SP_176034_trueColor_2023-01-19_day.tif
   [MEMORY] Initial: 693.8 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJEC

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/202302_Earthquake_Turkiye_LC08_L2SP_176034_trueColor_2023-01-19_day.tif
   [MEMORY] Final: 693.8 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC08_L2SP_176034_trueColor_2023-01-19_day.tif

✅ Batch processing complete: 12 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/files_converted.csv
📁 COGs saved locally to: output/202302_Earthquake_Turkiye

📊 BATCH PROCESS

# Landsat 8, naturalColor

In [20]:
# Define filename creator functions for different file types
pattern = re.compile(r'LC08.*naturalColor\.tif$')

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if pattern.search(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202302_Earthquake_Turkiye_LC08_L1TP_174034_naturalColor_2023-02-22_day.tif
  202302_Earthquake_Turkiye_LC08_L1TP_174035_naturalColor_2023-02-22_day.tif
  202302_Earthquake_Turkiye_LC08_L1TP_176034_naturalColor_2023-02-20_day.tif
  202302_Earthquake_Turkiye_LC08_L1TP_176035_naturalColor_2023-02-20_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_171034_naturalColor_2023-01-16_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_171035_naturalColor_2023-01-16_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172034_naturalColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172035_naturalColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172036_naturalColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174033_naturalColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174034_naturalColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174035_naturalColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174036_naturalCol

In [21]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename, 
                                target_dir = "Landsat/naturalColor", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202302_Earthquake_Turkiye_LC08_L1TP_174034_naturalColor_2023-02-22_day.tif
  202302_Earthquake_Turkiye_LC08_L1TP_174035_naturalColor_2023-02-22_day.tif
  202302_Earthquake_Turkiye_LC08_L1TP_176034_naturalColor_2023-02-20_day.tif
  202302_Earthquake_Turkiye_LC08_L1TP_176035_naturalColor_2023-02-20_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_171034_naturalColor_2023-01-16_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_171035_naturalColor_2023-01-16_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172034_naturalColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172035_naturalColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_172036_naturalColor_2023-02-08_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174033_naturalColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174034_naturalColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174035_naturalColor_2023-01-21_day.tif
  202302_Earthquake_Turkiye_LC08_L2SP_174036_naturalColor

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/202302_Earthquake_Turkiye_LC08_L1TP_174034_naturalColor_2023-02-22_day.tif
   [MEMORY] Final: 702.4 MB (Change: +8.3 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC08_L1TP_174034_naturalColor_2023-02-22_day.tif

[2/16] Processing: drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_174035_20230222_naturalColor.tif
   Output filename: 202302_Earthquake_Turkiye_LC08_L1TP_174035_naturalColor_2023-02-22_day.tif
   [MEMORY] Initial: 702.4 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache

   [BAND 2/3] Processing...


...

   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/202302_Earthquake_Turkiye_LC08_L2SP_175035_naturalColor_2023-02-13_day.tif
   [MEMORY] Final: 700.9 MB (Change: -22.8 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC08_L2SP_175035_naturalColor_2023-02-13_day.tif

[16/16] Processing: drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_176034_20230119_naturalColor.tif
   Output filename: 202302_Earthquake_Turkiye_LC08_L2SP_176034_naturalColor_2023-01-19_day.tif
   [MEMORY] Initial: 700.9 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cac

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/202302_Earthquake_Turkiye_LC08_L2SP_176034_naturalColor_2023-01-19_day.tif
   [MEMORY] Final: 702.5 MB (Change: +1.6 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC08_L2SP_176034_naturalColor_2023-01-19_day.tif

✅ Batch processing complete: 16 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/files_converted.csv
📁 COGs saved locally to: output/202302_Earthquake_Turkiye



In [23]:
keys

['drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_174034_20230222_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_174035_20230222_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_176034_20230220_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L1TP_176035_20230220_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171034_20230116_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171034_20230116_trueColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171035_20230116_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_171035_20230116_trueColor.tif',
 'drcs_activations/202302_Earthquake_Turkiye/landsat/landsat8/LC08_L2SP_172034_20230208_naturalColor.tif',
 'drcs_activations/202302_Earthquake_Turkiy

# Landsat 9, trueColor

In [24]:
# Define filename creator functions for different file types

pattern = re.compile(r'LC09.*trueColor\.tif$')

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if pattern.search(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202302_Earthquake_Turkiye_LC09_171034_trueColor_2023-01-24_day.tif
  202302_Earthquake_Turkiye_LC09_171035_trueColor_2023-01-24_day.tif
  202302_Earthquake_Turkiye_LC09_172033_trueColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172033_trueColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172033_trueColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172034_trueColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172034_trueColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172034_trueColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172035_trueColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172035_trueColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172035_trueColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172036_trueColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_173033_trueColor_2023-01-22_day.tif
  202302_Earthquake_Turkiye_LC09_173034_trueColor_2023-01-22_day.tif
  202302_Eart

In [25]:
# Process S1 WTR files

results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename, 
                                target_dir = "Landsat/trueColor", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202302_Earthquake_Turkiye_LC09_171034_trueColor_2023-01-24_day.tif
  202302_Earthquake_Turkiye_LC09_171035_trueColor_2023-01-24_day.tif
  202302_Earthquake_Turkiye_LC09_172033_trueColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172033_trueColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172033_trueColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172034_trueColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172034_trueColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172034_trueColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172035_trueColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172035_trueColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172035_trueColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172036_trueColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_173033_trueColor_2023-01-22_day.tif
  202302_Earthquake_Turkiye_LC09_173034_trueColor_2023-01-22_day.tif
  202302_Earthq

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/202302_Earthquake_Turkiye_LC09_171034_trueColor_2023-01-24_day.tif
   [MEMORY] Final: 702.9 MB (Change: +0.3 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC09_171034_trueColor_2023-01-24_day.tif

[2/29] Processing: drcs_activations/202302_Earthquake_Turkiye/landsat/landsat9/LC09_171035_20230124_trueColor.tif
   Output filename: 202302_Earthquake_Turkiye_LC09_171035_trueColor_2023-01-24_day.tif
   [MEMORY] Initial: 702.9 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG:

   [BAND 2/3] Processing...


...

   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/202302_Earthquake_Turkiye_LC09_175035_trueColor_2023-01-20_day.tif
   [MEMORY] Final: 740.8 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC09_175035_trueColor_2023-01-20_day.tif

[29/29] Processing: drcs_activations/202302_Earthquake_Turkiye/landsat/landsat9/LC09_176034_20230127_trueColor.tif
   Output filename: 202302_Earthquake_Turkiye_LC09_176034_trueColor_2023-01-27_day.tif
   [MEMORY] Initial: 740.8 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Converting to EPSG

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/202302_Earthquake_Turkiye_LC09_176034_trueColor_2023-01-27_day.tif
   [MEMORY] Final: 740.8 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC09_176034_trueColor_2023-01-27_day.tif

✅ Batch processing complete: 29 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/files_converted.csv
📁 COGs saved locally to: output/202302_Earthquake_Turkiye

📊 BATCH PROCESSING SUMMAR

# Landsat 9, naturalColor

In [27]:

pattern = re.compile(r'LC09.*naturalColor\.tif$')

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if pattern.search(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202302_Earthquake_Turkiye_LC09_171034_naturalColor_2023-01-24_day.tif
  202302_Earthquake_Turkiye_LC09_171035_naturalColor_2023-01-24_day.tif
  202302_Earthquake_Turkiye_LC09_172033_naturalColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172033_naturalColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172033_naturalColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172034_naturalColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172034_naturalColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172034_naturalColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172035_naturalColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172035_naturalColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172035_naturalColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172036_naturalColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_173033_naturalColor_2023-01-22_day.tif
  202302_Earthquake_Turkiye_LC09_173033_nat

In [28]:
# Define filename creator functions for different file types

# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename, 
                                target_dir = "Landsat/naturalColor", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202302_Earthquake_Turkiye_LC09_171034_naturalColor_2023-01-24_day.tif
  202302_Earthquake_Turkiye_LC09_171035_naturalColor_2023-01-24_day.tif
  202302_Earthquake_Turkiye_LC09_172033_naturalColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172033_naturalColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172033_naturalColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172034_naturalColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172034_naturalColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172034_naturalColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172035_naturalColor_2023-01-15_day.tif
  202302_Earthquake_Turkiye_LC09_172035_naturalColor_2023-01-31_day.tif
  202302_Earthquake_Turkiye_LC09_172035_naturalColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_172036_naturalColor_2023-02-16_day.tif
  202302_Earthquake_Turkiye_LC09_173033_naturalColor_2023-01-22_day.tif
  202302_Earthquake_Turkiye_LC09_173033_natur

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/202302_Earthquake_Turkiye_LC09_171034_naturalColor_2023-01-24_day.tif
   [MEMORY] Final: 742.1 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC09_171034_naturalColor_2023-01-24_day.tif

[2/34] Processing: drcs_activations/202302_Earthquake_Turkiye/landsat/landsat9/LC09_171035_20230124_naturalColor.tif
   Output filename: 202302_Earthquake_Turkiye_LC09_171035_naturalColor_2023-01-24_day.tif
   [MEMORY] Initial: 742.1 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Conv

   [BAND 2/3] Processing...


...

   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/202302_Earthquake_Turkiye_LC09_175035_naturalColor_2023-02-21_day.tif
   [MEMORY] Final: 753.9 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC09_175035_naturalColor_2023-02-21_day.tif

[34/34] Processing: drcs_activations/202302_Earthquake_Turkiye/landsat/landsat9/LC09_176034_20230127_naturalColor.tif
   Output filename: 202302_Earthquake_Turkiye_LC09_176034_naturalColor_2023-01-27_day.tif
   [MEMORY] Initial: 753.9 MB
   [DOWNLOAD] Downloading from S3...
   [DOWNLOAD] ✅ Saved to cache
   [REPROJECT] Con

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'deflate' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/202302_Earthquake_Turkiye_LC09_176034_naturalColor_2023-01-27_day.tif
   [MEMORY] Final: 753.9 MB (Change: +0.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202302_Earthquake_Turkiye_LC09_176034_naturalColor_2023-01-27_day.tif

✅ Batch processing complete: 34 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/files_converted.csv
📁 COGs saved locally to: output/202302_Earthquake_Turkiye

📊 BATCH PR

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [13]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")


📊 Memory Usage Summary:
  Current memory usage: 1166.1 MB
  Available memory: 28454.5 MB
  Memory percent used: 10.0%
